## Forward Kinematics (symbolic, from an RTB MDH robot)
1. Enter the robot with `DHRobot` / `RevoluteMDH` / `PrismaticMDH` (same as Lab 1)
2. Print each frame transform ${}^{i-1}T_{i}$
3. Print the full forward kinematics ${}^{0}T_{e}$

In [17]:
import numpy as np
import sympy as sp
from IPython.display import display, Math
from roboticstoolbox import DHRobot, RevoluteMDH, PrismaticMDH
from spatialmath import SE3

def Rx(t): return sp.Matrix([[1, 0, 0, 0], [0, sp.cos(t), -sp.sin(t), 0], [0, sp.sin(t), sp.cos(t), 0], [0, 0, 0, 1]])
def Rz(t): return sp.Matrix([[sp.cos(t), -sp.sin(t), 0, 0], [sp.sin(t), sp.cos(t), 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
def Tx(d): return sp.Matrix([[1, 0, 0, d], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
def Tz(d): return sp.Matrix([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, d], [0, 0, 0, 1]])

# Modified DH (Craig): T^{i-1}_i = Rx(alpha_{i-1}) Tx(a_{i-1}) Rz(theta_i) Tz(d_i)
def mdh(a, alpha, d, theta):
    return Rx(alpha) * Tx(a) * Rz(theta) * Tz(d)

# Standard DH: T^{i-1}_i = Rz(theta_i) Tz(d_i) Tx(a_i) Rx(alpha_i)
def dh(a, alpha, d, theta):
    return Rz(theta) * Tz(d) * Tx(a) * Rx(alpha)

# float -> clean sympy number: np.pi/2 -> pi/2, 6e-17 -> 0
# with lengths={'l1': 0.64, ...}: 0.64 -> l1, -0.2 -> -l2
def num(x, lengths=None):
    x = float(x)
    if abs(x) < 1e-9:
        return sp.Integer(0)
    for name, value in (lengths or {}).items():
        if np.isclose(abs(x), value):
            return sp.sign(x) * sp.Symbol(name, positive=True)
    s = sp.nsimplify(x, [sp.pi], tolerance=1e-9)
    return s if s.has(sp.pi) or s.is_integer else sp.Float(round(x, 6))

# 1.0*q3 -> q3, -1.0*sin(q1) -> -sin(q1)
def clean(T):
    return T.applyfunc(lambda e: e.replace(lambda x: x.is_Float and float(x).is_integer(), lambda x: sp.Integer(int(x))))

# display only: sin(q1) -> s_1, cos(q1 + q2) -> c_12
def short(T):
    def sub(f):
        terms = sp.Add.make_args(f.args[0])
        if not all(t.is_Symbol and t.name[0] == 'q' for t in terms):
            return f
        idx = ''.join(sorted((t.name[1:] for t in terms), key=int))
        return sp.Symbol(('s' if f.func == sp.sin else 'c') + '_{' + idx + '}')
    return T.applyfunc(lambda e: e.replace(lambda x: x.func in (sp.sin, sp.cos), sub))

# display only: short() + group terms with the same s/c factor
# c1*c2*l4 + c1*c2*(l3 + q3) -> c1*c2*(l3 + l4 + q3)
def simple(T):
    def group(e):
        groups = {}
        for t in sp.Add.make_args(sp.expand(e)):
            trig = sp.Mul(*[f for f in sp.Mul.make_args(t) if f.is_Symbol and f.name[:2] in ('s_', 'c_')])
            groups[trig] = groups.get(trig, 0) + t / trig
        return sp.Add(*[k * v for k, v in groups.items()])
    return short(T).applyfunc(group)

### 1 - Enter your robot
Same syntax as Lab 1. Joint variables become `q1, q2, ...` automatically.
Every length listed in `link_lengths` is shown as its symbol (`l1`, `l2`, ...). Give each length a different value, or two lengths will print as the same symbol.

In [18]:
# edit here: link lengths (name -> value in m)
l1 = 0.64
l2 = 0.2
l3 = 0.22
l4 = 0.095
link_lengths = {'l1': l1, 'l2': l2, 'l3': l3, 'l4': l4}

# edit here: define your robot using the MDH convention
robot_mdh = DHRobot([
    RevoluteMDH(a=0.0, alpha=0.0,     d=l1),
    RevoluteMDH(a=0.0, alpha=np.pi/2, d=-l2, offset=np.pi/2),
    PrismaticMDH(a=0.0, alpha=np.pi/2, offset=l3),
], name='my_mdh')

# edit here: set the tool transform to match your robot's end-effector
robot_mdh.tool = SE3.Tz(l4)

print(robot_mdh)

DHRobot: my_mdh, 3 joints (RRP), modified DH parameters
┌──────┬───────┬───────────┬────────────┐
│ aⱼ₋₁ │ ⍺ⱼ₋₁  │    θⱼ     │     dⱼ     │
├──────┼───────┼───────────┼────────────┤
│  0.0 │  0.0° │        q1 │       0.64 │
│  0.0 │ 90.0° │  q2 + 90° │       -0.2 │
│  0.0 │ 90.0° │      0.0° │  q3 + 0.22 │
└──────┴───────┴───────────┴────────────┘

┌──────┬───────────────────────────────────────┐
│ tool │ t = 0, 0, 0.095; rpy/xyz = 0°, 0°, 0° │
└──────┴───────────────────────────────────────┘



### Symbolic MDH table (generated from `robot_mdh`)

In [19]:
n = robot_mdh.n
q = sp.symbols(f'q1:{n + 1}', real=True)

# (a, alpha, d, theta) per joint, joint variable + offset put in theta (R) or d (P)
table = []
for qi, L in zip(q, robot_mdh.links):
    if L.isprismatic:
        d, theta = qi + num(L.offset, link_lengths), num(L.theta)
    else:
        d, theta = num(L.d, link_lengths), qi + num(L.offset)
    table.append((num(L.a, link_lengths), num(L.alpha), d, theta))

T_tool = sp.Matrix(robot_mdh.tool.A).applyfunc(num)
T_tool[0:3, 3] = T_tool[0:3, 3].applyfunc(lambda v: num(v, link_lengths))

a_h, al_h = (r'a_{i-1}', r'\alpha_{i-1}') if robot_mdh.mdh else (r'a_i', r'\alpha_i')
rows = r' \\ '.join(' & '.join(sp.latex(v) for v in (i, *row)) for i, row in enumerate(table, start=1))
display(Math(rf'\begin{{array}}{{c|cccc}} i & {a_h} & {al_h} & d_i & \theta_i \\ \hline ' + rows + r' \end{array}'))

<IPython.core.display.Math object>

### 2 - Transform of each frame ${}^{i-1}T_{i}$

In [20]:
link_T = mdh if robot_mdh.mdh else dh

T_frames = []
for i, row in enumerate(table, start=1):
    Ti = clean(sp.simplify(link_T(*row)))
    T_frames.append(Ti)
    display(Math(rf'{{}}^{{{i-1}}}T_{{{i}}} = ' + sp.latex(simple(Ti))))

display(Math(rf'{{}}^{{{n}}}T_{{e}} = ' + sp.latex(T_tool)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 3 - Forward kinematics ${}^{0}T_{e}$

In [21]:
T0e = sp.Matrix(robot_mdh.base.A).applyfunc(num)
for Ti in T_frames:
    T0e = T0e * Ti
T0e = clean(sp.simplify(T0e * T_tool))

display(Math(r'{}^{0}T_{e} = ' + sp.latex(simple(T0e))))
display(Math(r'p_e = ' + sp.latex(simple(T0e[0:3, 3]))))
display(Math(r'R_e = ' + sp.latex(simple(T0e[0:3, 0:3]))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Numeric example
Put joint values into every ${}^{i-1}T_{i}$ and into ${}^{0}T_{e}$, then compare with RTB `fkine`.

In [ ]:
# helpers: plug numbers into a symbolic result and print it rounded
def val(e, subs):
    return float(sp.sympify(e).subs(subs).evalf())
def mat(e, subs, d=4):
    return sp.Matrix(e).subs(subs).evalf().applyfunc(lambda v: sp.Float(round(float(v), d)) if abs(float(v)) > 1e-12 else 0)
def showv(lhs, v, unit=''):
    display(Math(rf'{lhs} = {v:.4f}\ \text{{{unit}}}'))
def showa(lhs, v):
    display(Math(rf'{lhs} = {v:.4f}\ \text{{rad}} = {np.rad2deg(v):.2f}^\circ'))
def showm(lhs, M):
    display(Math(lhs + ' = ' + sp.latex(M)))

# edit here: joint values (deg for revolute, m for prismatic)
q_in = [30, 30, 0.18]

q_val = [v if L.isprismatic else np.deg2rad(v) for v, L in zip(q_in, robot_mdh.links)]
nums = {sp.Symbol(k, positive=True): v for k, v in link_lengths.items()}
nums.update(dict(zip(q, q_val)))

for i, Ti in enumerate(T_frames, start=1):
    showm(rf'{{}}^{{{i-1}}}T_{{{i}}}', mat(Ti, nums))
showm(rf'{{}}^{{{n}}}T_{{e}}', mat(T_tool, nums))

T0e_num = mat(T0e, nums)
showm(r'{}^{0}T_{e}', T0e_num)
showm(r'p_e', T0e_num[0:3, 3])

T_rtb = robot_mdh.fkine(q_val).A
print('|T0e - RTB fkine| =', f'{np.linalg.norm(np.array(T0e.subs(nums).evalf(), dtype=float) - T_rtb):.1e}')

### Same with Robotics Toolbox
Uses the same `q_in` as the numeric example.
The cell declares its own MDH robot first, so it can be copied and run on its own (lengths in m: l1 = 0.64, l2 = 0.2, l3 = 0.22, l4 = 0.095).


In [ ]:
from roboticstoolbox import DHRobot, RevoluteMDH, PrismaticMDH
from spatialmath import SE3

# 1) declare the robot with MDH (a = a_{i-1}, alpha = alpha_{i-1}, d = d_i, offset added to the joint variable)
robot = DHRobot([
    RevoluteMDH(a=0.0, alpha=0.0,     d=0.64,                    qlim=[-np.pi, np.pi]),   # joint 1 (R)
    RevoluteMDH(a=0.0, alpha=np.pi/2, d=-0.2,  offset=np.pi/2,   qlim=[-np.pi, np.pi]),   # joint 2 (R)
    PrismaticMDH(a=0.0, alpha=np.pi/2, offset=0.22,              qlim=[0.0, 0.3]),        # joint 3 (P), m
], name='my_mdh')
robot.tool = SE3.Tz(0.095)                  # l4: last joint -> tip
print(robot)

# 2) use it
T = robot.fkine(q_val)                  # SE3 of the tip
print('T0e =\n', np.round(T.A, 4))
print('p_e         =', np.round(T.t, 4))
print('R_e =\n', np.round(T.R, 4))
print('rpy (deg)   =', np.round(T.rpy(unit='deg'), 2))

# every frame: fkine_all gives {}^0T_i for i = 0..n (no tool)
for i, Ti in enumerate(robot.fkine_all(q_val)):
    print(f'p_{i} =', np.round(Ti.t, 4))